# Week 5a Hands-On Lab — Grounded or Hallucinated? Probing a VLM

**ESP3201 · formative hands-on lab.** Best on free-tier Colab with a **GPU (T4) runtime** for the local model, or use the **hosted-API** path. An **offline mock** path lets you validate the pipeline with no model at all.

You synthesize scenes with **exact ground truth** and probe a VLM three ways:
1. **Easy probe** — present vs random-absent objects (hallucination + recall).
2. **Hard probe** — *binding traps*: ask for a `red circle` when the scene has a red square and a blue circle (color and shape both present, but not together). VLMs hallucinate these far more.
3. **Prompt sensitivity** — the same question in different wordings; a grounded model answers all the same.

> **Report only what your run actually produced.** A capable model can be *well-behaved* on these clean shapes — if so, that is a finding about your evaluation inputs, not a pass.

**Attribution.** Present/absent probing follows **POPE** (Li et al., 2023). Binding-error and prompt-sensitivity framing are standard VLM-evaluation practice. Code is original to ESP3201.

## Setup

In [ ]:
import sys, os

COURSE_REPO_URL = "https://github.com/bingquan/esp3201-public.git"

os.system('pip install -q pillow')
try:
    import vlm_lab
except ModuleNotFoundError:
    cand = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    if os.path.exists(os.path.join(cand, 'vlm_lab.py')):
        sys.path.insert(0, cand)
    elif COURSE_REPO_URL:
        os.system(f'git clone -q {COURSE_REPO_URL} course_repo')
        sys.path.insert(0, 'course_repo/labs/week05_embodied_system_critique/starter')
    else:
        raise ModuleNotFoundError('vlm_lab.py not found. On Colab set COURSE_REPO_URL.')
    import vlm_lab
from vlm_lab import (synthesize_scene, build_probe, build_binding_probe,
                     run_probe, score, run_prompt_sensitivity,
                     score_prompt_sensitivity, run_probe_on_images,
                     load_labeled_images, MockVLM, HFVLM, GeminiVLM, VOCAB)
print('vlm_lab loaded. vocabulary size:', len(VOCAB))

## Look at a synthesized scene, an easy probe, and a binding trap

In [ ]:
img, present = synthesize_scene(seed=3, n_objects=3)
print('present (ground truth):', sorted(present))
print('easy probe :', build_probe(present, n_absent=3, seed=3))
print('binding trap probe:', build_binding_probe(present, seed=3))
img  # displays inline

## Choose a backend

Pick ONE and run its cell.

- **A. Offline mock** — no model; simulates a yes-bias. Good for a dry run.
- **B. Local VLM (T4)** — defaults to `HuggingFaceTB/SmolVLM-Instruct`, **verified to fit a free T4** (~5 GB; see `pinned_models.md`). Swap `MODEL_ID` and re-check memory.
- **C. Hosted API (free tier)** — a Gemini Flash model. **PIN THIS** and supply a key.

In [ ]:
# --- A. Offline mock ---
backend = MockVLM(yes_bias=0.4)   # try 0.0, 0.4, 0.9
print('using MockVLM')

In [ ]:
# --- B. Local VLM on T4 (uncomment to use) ---
# os.system('pip install -q transformers accelerate torch')
# MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'   # VERIFIED on T4: ~5 GB fp16
# backend = HFVLM(model_id=MODEL_ID)
# print('using HFVLM', MODEL_ID)

In [ ]:
# --- C. Hosted Gemini (free tier) (uncomment and PIN THIS) ---
# os.system('pip install -q google-generativeai')
# from google.colab import userdata
# os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
# GEMINI_MODEL = ''  # PIN THIS, a current free-tier Gemini Flash id
# backend = GeminiVLM(model_id=GEMINI_MODEL)
# print('using GeminiVLM', GEMINI_MODEL)

## Probe 1 + 2 — easy vs binding-trap

Compare hallucination on random-absent objects vs binding traps. If the model is honest on easy absents but hallucinates traps, you have found a binding failure.

In [ ]:
easy = score(run_probe(backend, n_scenes=6, hard=False))
hard = score(run_probe(backend, n_scenes=6, hard=True))
print('EASY (random absent):', easy)
print('HARD (binding traps):', hard)

## Probe 3 — prompt sensitivity

Ask the SAME object question four ways. `flip_rate` > 0 means the answer depends on wording, not just on the image.

In [ ]:
sens = run_prompt_sensitivity(backend, n_scenes=3, n_objects=3, n_absent=3)
print('PROMPT SENSITIVITY:', score_prompt_sensitivity(sens))
for r in [x for x in sens if x.flipped][:5]:
    print(f'  FLIP {r.obj} (present={r.present}) answers={r.answers}')

## Inspect the failures that did occur

In [ ]:
recs = run_probe(backend, n_scenes=6, hard=True)
halluc = [r for r in recs if (not r.present) and r.answered_yes]
missed = [r for r in recs if r.present and not r.answered_yes]
print(f'{len(halluc)} hallucinated (yes on absent):', [(r.scene_seed, r.obj) for r in halluc[:6]])
print(f'{len(missed)} missed (no on present):    ', [(r.scene_seed, r.obj) for r in missed[:6]])

## Optional — real images for a stronger hallucination signal

Clean synthetic shapes are *easy*; a good VLM may not hallucinate on them at all. To push harder, drop a few natural photos into a folder with a `labels.json` (`filename -> [present objects]`) and re-run the same metric:

```python
items = load_labeled_images('my_images')           # PIL images + ground truth
print(score(run_probe_on_images(backend, items)))
```

## Worksheet (your deliverable)

### 1. Grounding table

| Probe | Accuracy | Hallucination rate | Recall (present) | Yes-rate |
|-------|----------|--------------------|------------------|----------|
| easy | | | | |
| binding traps | | | | |

Prompt-sensitivity flip rate (overall / present / absent): ____ / ____ / ____

### 2. What failed, and what didn't

- Did the model **hallucinate** (yes on absent)? Did binding traps change that vs easy absents?
- Did it **miss present objects** (recall < 1)? Did its answer **flip with phrasing**?

### 3. The evaluation-input lesson

- If a failure mode did **not** appear, was that because the model is good, or because the synthetic shapes are too easy? What input *would* expose it? (Try the real-image cell.)

### 4. When to trust it

- Is the model yes-biased (hallucinates) or no-biased (misses real objects)? What does each imply for a robot's perception module?
- `One deployment guardrail you would add before trusting this VLM:`

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking